First version. Only valid ISBNs obtained by removing invalid characters (but result not checked for actual existence). Only non-zero ratings. Only users and books that have at least 3 ratings. 

Update: min_rats = 10

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

from load_data import load_data
from valid_test_select import valid_test_select
from initialize_model import initialize_mu_b_c
from helpers import get_rmse
from fit_model import fit_model_no_UV, fit_model_full

In [7]:
np.set_printoptions(linewidth=150, precision=6)  # 75, 8

# Loading Data

In [8]:
df = load_data()

In [9]:
df.agg(["min", "max", "nunique", "dtype", "count", "size"])

,userid,isbn,rating
min,2,0000001481,0
max,278854,9998150752,10
nunique,103371,328465,11
dtype,int32,object,int8
count,1135338,1135338,1135338
size,1135338,1135338,1135338


In [10]:
df = df[df.rating > 0]

In [11]:
df.agg(["min", "max", "nunique", "size"])

,userid,isbn,rating
min,8,000003827X,1
max,278854,9997555635,10
nunique,76331,179924,10
size,427261,427261,427261


# Creating Matrix Y

In [12]:
# Encoding userids and isbns as categories (integers 0 to N-1)

user_cats = df.userid.astype("category")
book_cats = df.isbn.astype("category")

In [13]:
# Creating sparse matrix and converting it to pd.DataFrame

Y_sparse = coo_matrix((df.rating.values, (user_cats.cat.codes, book_cats.cat.codes)))
Y = pd.DataFrame(Y_sparse.toarray())

In [14]:
# print("Y shape:", Y.shape)  # (76331, 179924)
# print("total entries:", (Y > 0).sum().sum())  # 427261
# print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))  # 5.6
# print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))  # 2.37

In [15]:
# Filtering out improbably active readers

# np.sort(np.partition((Y > 0).sum(axis=1), -20)[-20:])
Y = Y.loc[(Y > 0).sum(axis=1) < 2000, :]

In [16]:
# Filtering users and books with enough observations

min_rats = 10
old_rows, old_cols = Y.shape
while True:
    Y_pos = Y > 0  # type: ignore
    user_rats = Y_pos.sum(axis=1)  # type: ignore
    book_rats = Y_pos.sum(axis=0)  # type: ignore
    new_rows = (user_rats >= min_rats).sum()
    new_cols = (book_rats >= min_rats).sum()
    # print(new_rows, new_cols)
    Y = Y.loc[user_rats >= min_rats, book_rats >= min_rats]  # type: ignore
    if (old_rows == new_rows) and (old_cols == new_cols):
        break
    old_rows, old_cols = new_rows, new_cols

In [17]:
print("Y shape:", Y.shape)
print("total entries:", (Y > 0).sum().sum())
print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))
print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))

Y shape: (1759, 1885)
total entries: 38732
avg number of ratings per user: 22.02
avg number of ratings per book: 20.55


In [18]:
# Converting Y to numpy array

Y_columns = Y.columns.copy()
Y = Y.to_numpy()

In [19]:
def get_isbn(pos):
    """Given position of column in matrix Y, return the original ISBN."""
    return book_cats.cat.categories[Y_columns[pos]]

# Selecting validation and test sets

In [20]:
Y_train, valid_data, test_data = valid_test_select(Y, 5_000, 5_000, seed=123)

In [21]:
print("Y_train shape:", Y_train.shape)
print("total train entries:", (Y_train > 0).sum().sum())
print("avg. # of train entries per user:", round((Y_train > 0).sum(axis=1).mean(), 2))
print("avg. # of train entries per book:", round((Y_train > 0).sum(axis=0).mean(), 2))
print("users with no ratings:", ((Y_train > 0).sum(axis=1) == 0).sum())
print("books with no ratings:", ((Y_train > 0).sum(axis=0) == 0).sum())

Y_train shape: (1759, 1885)
total train entries: 28732
avg. # of train entries per user: 16.33
avg. # of train entries per book: 15.24
users with no ratings: 0
books with no ratings: 0


# ALS

In [22]:
Y_csr = csr_matrix(Y_train)
Y_csc = Y_csr.tocsc()

## Model with just a training global mean

In [ ]:
mu, _, _ = initialize_mu_b_c(Y_train)
print("test rmse of model=mu:", get_rmse(mu, test_data))
# test rmse of model=mu: 1.7398115597510162

test rmse of model=mu: 1.7398115597510162


## Model with only biases

In [24]:
lambda_bias = 4.5
mu, b, c = initialize_mu_b_c(Y_train)
mu, b, c, valid_rmse = fit_model_no_UV(
    Y_csr, Y_csc, mu, b, c, lambda_bias, valid_data, 1e-5, info=2
)

original valid rmse: 1.5411451555399538
counter=1, max_norm_diff=17.2153893752701, valid_rmse=1.490041959411712, valid_rmse_diff=0.05110319612824177
counter=2, max_norm_diff=2.848458345905925, valid_rmse=1.4848238802208134, valid_rmse_diff=0.005218079190898672
counter=3, max_norm_diff=0.41630479645038543, valid_rmse=1.4845112691920632, valid_rmse_diff=0.0003126110287501316
counter=4, max_norm_diff=0.15677662602254325, valid_rmse=1.4844816735081707, valid_rmse_diff=2.9595683892491564e-05
counter=5, max_norm_diff=0.12616243560434065, valid_rmse=1.4844759824162344, valid_rmse_diff=5.6910919363772905e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=6, max_norm_diff=0.10418971666943508, valid_rmse=1.4844729914137054, valid_rmse_diff=2.9910025289847653e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=6, max_norm_diff=0.10418971666943508, valid_rmse=1.4844729914137054


In [ ]:
test_preds = np.clip(mu + b[test_data.rows] + c[test_data.cols], 1, 10)
test_rmse = get_rmse(test_preds, test_data)
print("test rmse of a tuned model with only biases:", test_rmse)
# test rmse of a tuned model with only biases: 1.4960189893805045

test rmse of a tuned model with only biases: 1.4960189893805045


## Full model

### k=1

In [26]:
k = 1
lambda_bias = 4.5
lambda_fact = 24
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 1.8442241945966444
counter=1, max_norm_diff=42.001419211233255, valid_rmse=1.5102742912374894, valid_rmse_diff=0.3339499033591551
counter=2, max_norm_diff=9.109655852884497, valid_rmse=1.485439415065786, valid_rmse_diff=0.024834876171703435
counter=3, max_norm_diff=1.413093049557736, valid_rmse=1.4845601105102508, valid_rmse_diff=0.0008793045555350876
counter=4, max_norm_diff=0.22750340389530088, valid_rmse=1.4844906645917026, valid_rmse_diff=6.944591854818327e-05
counter=5, max_norm_diff=0.13038654533020455, valid_rmse=1.4844779997154596, valid_rmse_diff=1.2664876243073664e-05
counter=6, max_norm_diff=0.10715084081547532, valid_rmse=1.4844733222534927, valid_rmse_diff=4.67746196686214e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=0.08798553094200925, valid_rmse=1.4844705930923594, valid_rmse_diff=2.7291611333080112e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=0.08798553094

In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4960301531899136

test rmse of full model: 1.4960301531899136


### k=2

In [28]:
k = 2
lambda_bias = 4.5
lambda_fact = 18.5
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 2.0695560482084407
counter=1, max_norm_diff=59.00280690384095, valid_rmse=1.5296636292211827, valid_rmse_diff=0.539892418987258
counter=2, max_norm_diff=12.176354725294487, valid_rmse=1.486143040293866, valid_rmse_diff=0.043520588927316695
counter=3, max_norm_diff=1.8340510656402311, valid_rmse=1.484514256188317, valid_rmse_diff=0.0016287841055491192
counter=4, max_norm_diff=0.8269777469043214, valid_rmse=1.484432010600629, valid_rmse_diff=8.224558768787915e-05
counter=5, max_norm_diff=0.7697276152338555, valid_rmse=1.4844236906205945, valid_rmse_diff=8.31998003447687e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=6, max_norm_diff=0.5067514230535224, valid_rmse=1.4844324225940722, valid_rmse_diff=-8.731973477615895e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=6, max_norm_diff=0.5067514230535224, valid_rmse=1.4844324225940722


In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4956779758866023

test rmse of full model: 1.4956779758866023


### k=4

In [30]:
k = 4
lambda_bias = 4.6
lambda_fact = 25
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 2.5150455736410953
counter=1, max_norm_diff=83.92120758117814, valid_rmse=1.5733775031260895, valid_rmse_diff=0.9416680705150058
counter=2, max_norm_diff=17.657557514682225, valid_rmse=1.4862964018343034, valid_rmse_diff=0.08708110129178603
counter=3, max_norm_diff=2.7372523388711536, valid_rmse=1.4846541212280726, valid_rmse_diff=0.0016422806062308393
counter=4, max_norm_diff=0.42880335157720856, valid_rmse=1.4846225223187774, valid_rmse_diff=3.159890929516074e-05
counter=5, max_norm_diff=0.18457495986081282, valid_rmse=1.484619743373042, valid_rmse_diff=2.7789457355265057e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=6, max_norm_diff=0.0956719913839972, valid_rmse=1.4846190792118354, valid_rmse_diff=6.641612064761659e-07
one iteration's improvement in valid rmse is under rmse_thresh
counter=6, max_norm_diff=0.0956719913839972, valid_rmse=1.4846190792118354


In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4960350855881273

test rmse of full model: 1.4960350855881273


### k=8

In [32]:
k = 8
lambda_bias = 4.4
lambda_fact = 20
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 3.2312940981454656
counter=1, max_norm_diff=118.12546234347951, valid_rmse=1.6660823911225084, valid_rmse_diff=1.5652117070229572
counter=2, max_norm_diff=23.99481281751236, valid_rmse=1.4874373058918315, valid_rmse_diff=0.1786450852306769
counter=3, max_norm_diff=3.759458090784109, valid_rmse=1.4842919206584917, valid_rmse_diff=0.003145385233339848
counter=4, max_norm_diff=1.085578961536948, valid_rmse=1.4842083486345692, valid_rmse_diff=8.357202392250684e-05
counter=5, max_norm_diff=0.6015122965093228, valid_rmse=1.4841979451515663, valid_rmse_diff=1.0403483002852099e-05
counter=6, max_norm_diff=0.35181900822673257, valid_rmse=1.484200515103723, valid_rmse_diff=-2.5699521566391326e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=0.22265093094239174, valid_rmse=1.484202889635564, valid_rmse_diff=-2.3745318409318372e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=0.22265093094239

In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4956133144033328

test rmse of full model: 1.4956133144033328


### k=16

In [34]:
k = 16
lambda_bias = 4.3
lambda_fact = 14.7
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 4.245149934105252
counter=1, max_norm_diff=164.41427866136425, valid_rmse=1.8446290349038759, valid_rmse_diff=2.4005208992013767
counter=2, max_norm_diff=33.556971661552694, valid_rmse=1.4960674213685057, valid_rmse_diff=0.3485616135353702
counter=3, max_norm_diff=5.628756468627815, valid_rmse=1.4853242492037106, valid_rmse_diff=0.010743172164795034
counter=4, max_norm_diff=2.4103381925276914, valid_rmse=1.4847263431348992, valid_rmse_diff=0.0005979060688114934
counter=5, max_norm_diff=1.4439638441533005, valid_rmse=1.4845572078529679, valid_rmse_diff=0.00016913528193129856
counter=6, max_norm_diff=1.0238504350402002, valid_rmse=1.484439178102887, valid_rmse_diff=0.00011802975008090577
counter=7, max_norm_diff=0.7885723337298002, valid_rmse=1.4843766865982526, valid_rmse_diff=6.249150463433217e-05
counter=8, max_norm_diff=0.6361336215766149, valid_rmse=1.4843572305401442, valid_rmse_diff=1.9456058108380248e-05
counter=9, max_norm_diff=0.5300690151104874, valid_rmse

In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.49540682027648

test rmse of full model: 1.49540682027648


### k=32

In [36]:
k = 32
lambda_bias = 4.3
lambda_fact = 13.7
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 5.818489488544072
counter=1, max_norm_diff=233.69736395408643, valid_rmse=2.1163609993933874, valid_rmse_diff=3.702128489150685
counter=2, max_norm_diff=45.1262906490908, valid_rmse=1.5105067424651955, valid_rmse_diff=0.605854256928192
counter=3, max_norm_diff=8.212362318444564, valid_rmse=1.486004181829424, valid_rmse_diff=0.024502560635771387
counter=4, max_norm_diff=3.2118967669838887, valid_rmse=1.4844500360065487, valid_rmse_diff=0.0015541458228753857
counter=5, max_norm_diff=1.8441254931488704, valid_rmse=1.4842313180319953, valid_rmse_diff=0.00021871797455341735
counter=6, max_norm_diff=1.3002003896145768, valid_rmse=1.4842351429938139, valid_rmse_diff=-3.824961818610362e-06
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=1.0081109669691157, valid_rmse=1.484313648775362, valid_rmse_diff=-7.850578154822152e-05
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=1.0081109669691157, val

In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4949407130934727

test rmse of full model: 1.4949407130934727


### k=256

In [38]:
k = 256
lambda_bias = 4
lambda_fact = 10
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 16.205800979007797
counter=1, max_norm_diff=668.1989140175673, valid_rmse=4.525100247468829, valid_rmse_diff=11.68070073153897
counter=2, max_norm_diff=115.2465750019616, valid_rmse=1.8450671218644794, valid_rmse_diff=2.680033125604349
counter=3, max_norm_diff=26.572596474088993, valid_rmse=1.5243378708498436, valid_rmse_diff=0.3207292510146358
counter=4, max_norm_diff=9.428640596391363, valid_rmse=1.4867928762773854, valid_rmse_diff=0.03754499457245819
counter=5, max_norm_diff=5.0269792475893675, valid_rmse=1.4830032190441373, valid_rmse_diff=0.0037896572332480893
counter=6, max_norm_diff=3.2653149452845343, valid_rmse=1.4831214338504943, valid_rmse_diff=-0.00011821480635698656
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=2.389891023585565, valid_rmse=1.4837059659504261, valid_rmse_diff=-0.0005845320999318115
one iteration's improvement in valid rmse is under rmse_thresh
counter=7, max_norm_diff=2.389891023585565, valid_r

In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4928940513717075

test rmse of full model: 1.4928940513717075


### k=512

In [40]:
k = 512
lambda_bias = 3.9
lambda_fact = 10
mu, b, c = initialize_mu_b_c(Y_train)
rng = np.random.default_rng(seed=123)
U = rng.normal(0, 1, size=(Y.shape[0], k))
V = rng.normal(0, 1, size=(Y.shape[1], k))
mu, b, c, U, V, valid_rmse = fit_model_full(
    Y_csr, Y_csc, mu, b, c, U, V, lambda_bias, lambda_fact, valid_data, 1e-5, info=2
)

original valid rmse: 22.448422884711615
counter=1, max_norm_diff=946.6972499553551, valid_rmse=6.152082515129562, valid_rmse_diff=16.296340369582055
counter=2, max_norm_diff=165.317604363761, valid_rmse=2.1126223952104093, valid_rmse_diff=4.039460119919153
counter=3, max_norm_diff=36.81185721440954, valid_rmse=1.5684771286765768, valid_rmse_diff=0.5441452665338324
counter=4, max_norm_diff=11.643256561113438, valid_rmse=1.4945636924297347, valid_rmse_diff=0.0739134362468421
counter=5, max_norm_diff=5.919519889711756, valid_rmse=1.4848695440847244, valid_rmse_diff=0.009694148345010367
counter=6, max_norm_diff=3.6113594816171806, valid_rmse=1.4836281821001962, valid_rmse_diff=0.0012413619845281776
counter=7, max_norm_diff=2.5637313785202633, valid_rmse=1.4837624548357216, valid_rmse_diff=-0.00013427273552535368
one iteration's improvement in valid rmse is under rmse_thresh
counter=8, max_norm_diff=1.9659712502189546, valid_rmse=1.4841797064039237, valid_rmse_diff=-0.00041725156820215936
o

In [ ]:
biases = mu + b[test_data.rows] + c[test_data.cols]
preds = np.clip(biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10)
print("test rmse of full model:", get_rmse(preds, test_data))
# test rmse of full model: 1.4941077611267521

test rmse of full model: 1.4941077611267521


In [ ]:
# global mean                           test_rmse = 1.7398115597510162
# only biases                           test_rmse = 1.4960189893805045
# k = 1,   l_bias = 4.5, l_fact = 24,   test_rmse = 1.4960301531899136
# k = 2,   l_bias = 4.5, l_fact = 18.5, test_rmse = 1.4956779758866023
# k = 4,   l_bias = 4.6, l_fact = 25,   test_rmse = 1.4960350855881273
# k = 8,   l_bias = 4.4, l_fact = 20,   test_rmse = 1.4956133144033328
# k = 16,  l_bias = 4.3, l_fact = 14.7, test_rmse = 1.49540682027648
# k = 32,  l_bias = 4.3, l_fact = 13.7, test_rmse = 1.4949407130934727
# k = 256, l_bias = 4,   l_fact = 10    test_rmse = 1.4928940513717075
# k = 512, l_bias = 3.9, l_fact = 10,   test_rmse = 1.4941077611267521